# Adaptive RAG for Multi-Hop Question Answering (Kaggle Runner)

This notebook runs the official benchmark pipeline, trains the **Evidence-State Marginal Retrieval Controller** (PyTorch Neural Residual Router & GBDT), and evaluates all baselines (B0 to B6) on Kaggle with GPU acceleration.

## 1. Install & Verify Dependencies
Kaggle comes with `torch`, `pandas`, `scikit-learn`, `numpy`, and `matplotlib` pre-installed. We only need `faiss-cpu` and `sentence-transformers`.

In [ ]:
!pip install -q faiss-cpu sentence-transformers pyyaml tqdm

## 2. Check GPU & Environment

In [ ]:
import torch
import os
import sys

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")

# Find the uploaded folder dynamically
import glob
candidates = glob.glob('/kaggle/input/**/RAG-OCT-27', recursive=True)
if candidates:
    src_dir = candidates[0]
    dst_dir = '/kaggle/working/RAG-OCT-27'
    if not os.path.exists(dst_dir):
        shutil.copytree(src_dir, dst_dir)
    os.chdir(dst_dir)
print('Current Working Directory:', os.getcwd())

## 3. Run Diagnostic Verification Suite
Audits tensor dimensions `(B, 7) -> (B, 1)`, gradient flow, Focal Loss penalty, and stability.

In [ ]:
!python tests/test_model_diagnostics.py

## 4. Train the Evidence Controller (Zero Data Leakage)
Trains on the official 700-query training split and tunes probability threshold $\tau=0.3$ on validation.

In [ ]:
# Train PyTorch Neural Residual Controller
!python scripts/train_controller.py --backend neural --epochs 50 --tau 0.3

# Or Train GBDT Decision Tree Controller
# !python scripts/train_controller.py --backend gbdt --tau 0.3

## 5. Master Evaluation of All Methods (B0 - B6)
Evaluates on the frozen unseen test split (150 queries) and full 1,000 queries.

In [ ]:
!python src/common/cache_builder.py
!python scripts/evaluate_all_methods.py

## 6. Display Benchmark Tables & Results

In [ ]:
import pandas as pd

test_summary = pd.read_csv("results/tables/test_split_summary.csv")
print("=== FROZEN TEST SPLIT (N=150 UNSEEN QUERIES) ===")
display(test_summary)

## 7. Run Essential Ablations & Generate Figures

In [ ]:
!python scripts/run_ablations.py
!python scripts/make_figures.py

## 8. View Generated Pareto Frontier Plots

In [ ]:
from IPython.display import Image, display

for fig_name in ["fig1_quality_vs_tokens.png", "fig2_quality_vs_latency.png", "fig3_tau_pareto_frontier.png"]:
    fig_path = os.path.join("figures", fig_name)
    if os.path.exists(fig_path):
        print(f"Displaying {fig_name}:")
        display(Image(filename=fig_path))
        print("-" * 60)